# Module 1: Apache Iceberg Fundamentals

This notebook contains code examples for Module 1 videos.

## Setup

Initialize Spark session for Module 1 examples.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Module 1 - Apache Iceberg Fundamentals") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

## Video 1: The Open Lakehouse

In [ ]:
# Show Spark configuration for connecting to the Catalog and Storage
# This demonstrates how the query engine connects to other Lakehouse components

import os

spark_conf_path = os.path.expanduser('~/.sparkconf/spark-defaults.conf')

with open(spark_conf_path, 'r') as f:
    spark_config = f.read()
    print("Spark Configuration (spark-defaults.conf):")
    print("=" * 80)
    print(spark_config)

## Video 2: Modeling Data into an Apache Iceberg Table

In [ ]:
# Create an unpartitioned table using CREATE TABLE AS SELECT (CTAS)
# This copies the NYC Taxi data from Parquet into an Iceberg table without partitioning

spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_unpartitioned
    USING iceberg
    TBLPROPERTIES (
        'write.target-file-size-bytes' = '16777216'
    )
    AS SELECT * FROM parquet.`s3a://warehouse/raw/`
""")

print("Unpartitioned table created!")
print("Location: s3a://warehouse/taxi/trips_unpartitioned/")

In [ ]:
# Query metadata tables to see what files are currently in the table
# Metadata tables are virtual tables built from metadata.json, manifest lists, and manifests
# They allow us to inspect the table structure without reading data files

print("Files Metadata Table:")
print("=" * 80)
spark.sql("""
    SELECT 
        file_path,
        file_size_in_bytes / 1024 / 1024 as size_mb,
        record_count,
        partition
    FROM polaris.taxi.trips_unpartitioned.files
""").show(truncate=False)

print("\nTable Summary:")
spark.sql("""
    SELECT 
        COUNT(*) as total_files,
        SUM(record_count) as total_records,
        ROUND(SUM(file_size_in_bytes) / 1024 / 1024, 2) as total_size_mb
    FROM polaris.taxi.trips_unpartitioned.files
""").show()

## Video 3: Hidden Partitioning in Apache Iceberg


In [ ]:
# Create a table with hidden partitioning using the days() transform
# This partitions by the epoch day (days since January 1, 1970) without storing a separate column

spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_by_day
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    TBLPROPERTIES (
        'write.target-file-size-bytes' = '16777216'
    )
    AS SELECT * FROM parquet.`s3a://warehouse/raw/`
""")

print("Daily partitioned table created!")
print("Partitioned by: days(tpep_pickup_datetime)")
print("\nNote: The partition transform is 'hidden' - no separate day column is stored")

In [ ]:
# Query with timestamp filter to demonstrate automatic partition pruning
# Even though we filter on the raw timestamp, Iceberg understands the partition transform

result = spark.sql("""
    SELECT 
        COUNT(*) as total_trips,
        ROUND(AVG(total_amount), 2) as avg_fare,
        MIN(tpep_pickup_datetime) as first_pickup,
        MAX(tpep_pickup_datetime) as last_pickup
    FROM polaris.taxi.trips_by_day
    WHERE tpep_pickup_datetime >= '2023-08-01'
      AND tpep_pickup_datetime < '2023-09-01'
""")

print("Query Results (August 2023 only):")
result.show()

print("\n" + "=" * 80)
print("Check Spark UI (http://localhost:4040/SQL/) to see:")
print("  - number of result data files: Only files for August (~31 files)")
print("  - number of skipped data files: Files from other months that were skipped")
print("  - Iceberg performed partition pruning without explicitly filtering on day column")

In [ ]:
# View column metrics using the files metadata table with readable_metrics
# This shows min/max values, null counts, and other statistics for each data file

print("Files Metadata with Column Metrics:")
print("=" * 80)

spark.sql("""
    SELECT 
        file_path,
        partition,
        record_count,
        ROUND(file_size_in_bytes / 1024 / 1024, 2) as size_mb,
        readable_metrics.PULocationID.lower_bound as PULocationID_min,
        readable_metrics.PULocationID.upper_bound as PULocationID_max,
        readable_metrics.total_amount.lower_bound as total_amount_min,
        readable_metrics.total_amount.upper_bound as total_amount_max
    FROM polaris.taxi.trips_by_day.files
    ORDER BY partition DESC
    LIMIT 10
""").show(truncate=False)

print("\n" + "=" * 80)
print("These metrics enable predicate pushdown:")
print("  - Files can be skipped if their min/max ranges don't overlap with query filters")
print("  - Metrics are collected from Parquet file footers and stored in manifest files")
print("  - Query planning uses these metrics WITHOUT opening data files")